In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import numpy as np
import pandas as pd
import pingouin as pg 
import statsmodels.api as sm
from natsort import natsorted
import plotly.graph_objects as go
from os.path import join as pjoin
from statsmodels.formula.api import ols
from numpy.random import MT19937, SeedSequence, RandomState

sys.path.append("../../")
import circletrack_behavior as ctb
import plotting_functions as pf

In [ ]:
## Settings
parent_dir = 'CircleTrack_Opto'
experiment_dir = 'RT_Opto1'
lin_path = f'../../../{parent_dir}/{experiment_dir}/output/lin_behav/'
circle_path = f'../../../{parent_dir}/{experiment_dir}/output/behav/'
fig_path = f'../../../{parent_dir}/{experiment_dir}/intermediate_figures'
maze_info = pd.read_csv(f'../../../{parent_dir}/{experiment_dir}/maze_yml/maze_info.csv')
chance_color = '#7d7d7d'
avg_color = 'midnightblue'
subject_color = 'darkgrey'
two_group_colors = ['midnightblue', 'darkorchid']
group_colors_dict = {'stGtACR2': 'darkorchid', 'mCherry': 'midnightblue'}
error_dict = {'stGtACR2': 'rgba(153,50,204,0.4)', 'mCherry': 'rgba(0,41,102,0.4)'}
excluded_mice = ['rto05', 'rto12'] ## rto12 excluded due to being put in the incorrect context, rto05 excluded because they could not learn A or B
group_list = ['mCherry', 'stGtACR2']
symbols_list = ['circle', 'diamond']

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

In [ ]:
## Randomize port numbers for each context for each mouse
rs = RandomState(MT19937(SeedSequence(1)))
context_list = ['A', 'B', 'C', 'D']
port_list = [0, 1, 2, 3]
mouse_list = [f'rto0{x}' for x in np.arange(2, 10)] + ['rto10', 'rto11', 'rto12']
potential_combinations = [
    [['reward1', 'reward5'], ['reward2', 'reward6'], ['reward3', 'reward7'], ['reward4', 'reward8']],
    [['reward1', 'reward6'], ['reward2', 'reward5'], ['reward3', 'reward8'], ['reward4', 'reward7']],
    [['reward1', 'reward4'], ['reward2', 'reward7'], ['reward3', 'reward6'], ['reward5', 'reward8']],
    [['reward1', 'reward5'], ['reward2', 'reward6'], ['reward3', 'reward8'], ['reward4', 'reward7']],
]

output = {}
for mouse in mouse_list:
    context_ports = {'A': [], 'B': [], 'C': [], 'D': []}
    randcont = rs.randint(0, len(context_list))
    randports = rs.choice(port_list, size=4, replace=False)
    for context, ports in zip(context_list, randports):
        context_ports[context].append(potential_combinations[randcont][ports])
    output[f'{mouse}'] = context_ports
port_df = pd.DataFrame(output)
port_df

### Plot example linear position curves with opto times overlaid.

In [ ]:
## Select mouse and day
mouse = 'rto08'
day = 16
session = f'{mouse}_{day}.feat'
behav = pd.read_feather(pjoin(circle_path, f'{mouse}/{session}'))

fig = pf.custom_graph_template(x_title='Time (s)', y_title='Degree', width=1000)
fig.add_trace(go.Scattergl(x=behav['t'], y=behav['a_pos'], mode='lines', line_color='darkgrey', 
                           line_width=2, showlegend=False))
fig.add_trace(go.Scattergl(x=behav['t'][behav['lick_port'] != -1], y=behav['a_pos'][behav['lick_port'] != -1], 
                           mode='markers', marker_color='black', name='Licks', opacity=0.6, marker_size=5))
fig.add_trace(go.Scattergl(x=behav['t'][behav['water']], y=behav['a_pos'][behav['water']], opacity=0.6,
                           mode='markers', marker_color='darkorchid', name='Rewards', marker_size=9))

## Plot opto events
opto_times = behav['t'][behav['opto_active']].to_numpy()
shifted_opto = behav.index[behav['opto_active']].diff().values
shifted_opto_end = shifted_opto != 1
shifted_opto_end = np.roll(shifted_opto_end, -1)
opto_start = opto_times[shifted_opto != 1]
opto_end = opto_times[shifted_opto_end]
for start, end in zip(opto_start, opto_end):
    fig.add_vrect(x0=start, x1=end, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.update_yaxes(range=[-10, 370])
fig.update_xaxes(range=[0, 900])
fig.show()
fig.write_image(pjoin(fig_path, f'{mouse}_{day}_lin_pos.png'), width=1000, height=500)

### Circle track lick accuracy and rewards.

In [ ]:
circletrack_results = {'mouse': [], 'day': [], 'sex': [], 'group': [], 'session': [], 'lick_accuracy': [], 'rewards': [], 
                       'num_licks': [], 'opto_time': [], 'correct_dir': []}
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass 
    else:
        mouse_path = pjoin(circle_path, mouse)
        sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
        group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
        for idx, session in enumerate(natsorted(os.listdir(mouse_path))):
            behav = pd.read_feather(pjoin(mouse_path, f'{session}'))
            behav = behav[~behav['probe']] ## exclude probe
            reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
            pc_thresh5 = ctb.lick_accuracy(behav, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)

            ## Get amount of opto stimulation
            
            if session == 'rto11_10.feat': ## maze leaked and ruined behavior for this day
                pass 
            else:
                circletrack_results['mouse'].append(mouse)
                circletrack_results['day'].append(idx + 1)
                circletrack_results['sex'].append(sex)
                circletrack_results['group'].append(group)
                circletrack_results['session'].append(np.unique(behav['session_two'])[0])
                circletrack_results['lick_accuracy'].append(pc_thresh5)
                circletrack_results['rewards'].append(np.sum(behav['water']))
                circletrack_results['num_licks'].append(behav[behav['lick_port'] != -1].shape[0])
                circletrack_results['opto_time'].append(np.sum(behav['opto_active']) * np.nanmean(behav['t'].diff())) ## in seconds
                circletrack_results['correct_dir'].append((np.sum(behav['correct_dir']) / behav.shape[0]) * 100) ## as percent
ct_df = pd.DataFrame(circletrack_results)

In [ ]:
## Plot lick accuracy across days
fig = pf.plot_behavior_across_days(ct_df[ct_df['day'] < 21], x_var='day', y_var='lick_accuracy', groupby_var=['day', 'group'], plot_transitions=[5.5, 10.5, 15.5], 
                                   transition_color=['darkgrey', 'darkgrey', 'darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=True, symbols=symbols_list,
                                   plot_datapoints=True, x_title='Day', y_title='Lick Accuracy (%)', titles=[''], height=500, width=650)
fig.add_vrect(x0=15.5, x1=16.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.add_vrect(x0=19.5, x1=20.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.update_yaxes(range=[0, 100])
fig.update_traces(name=f'mCherry ({ct_df['mouse'][ct_df['group'] == 'mCherry'].unique().shape[0]})',
                  selector={'name': 'mCherry'})
fig.update_traces(name=f'stGtACR2 ({ct_df['mouse'][ct_df['group'] == 'stGtACR2'].unique().shape[0]})',
                  selector={'name': 'stGtACR2'})
fig.show()
fig.write_image(pjoin(fig_path, 'lick_accuracy.png'), width=650, height=500)

## T-test for day 16 behavior
data = ct_df[ct_df['day'] == 16].reset_index(drop=True)
model = ols('lick_accuracy ~ C(group)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=3)
print(anova_table)

In [ ]:
## Plot lick accuracy across days with reversal
fig = pf.plot_behavior_across_days(ct_df, x_var='day', y_var='lick_accuracy', groupby_var=['day', 'group'], plot_transitions=[5.5, 10.5, 15.5, 20.5], 
                                   transition_color=['darkgrey', 'darkgrey', 'darkgrey', 'darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=True, symbols=symbols_list,
                                   plot_datapoints=True, x_title='Day', y_title='Lick Accuracy (%)', titles=[''], height=500, width=650)
fig.add_vrect(x0=15.5, x1=16.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.add_vrect(x0=19.5, x1=20.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.add_vrect(x0=20.5, x1=21.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.update_yaxes(range=[0, 100])
fig.update_traces(name=f'mCherry ({ct_df['mouse'][ct_df['group'] == 'mCherry'].unique().shape[0]})',
                  selector={'name': 'mCherry'})
fig.update_traces(name=f'stGtACR2 ({ct_df['mouse'][ct_df['group'] == 'stGtACR2'].unique().shape[0]})',
                  selector={'name': 'stGtACR2'})
fig.show()
fig.write_image(pjoin(fig_path, 'lick_accuracy_with_reversal.png'), width=650, height=500)

## T-test for day 16 behavior
data = ct_df[ct_df['day'] == 16].reset_index(drop=True)
model = ols('lick_accuracy ~ C(group)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=3)
print(anova_table)

In [ ]:
## Plot lick accuracy across days up to day 16
fig = pf.plot_behavior_across_days(ct_df[ct_df['day'] < 17], x_var='day', y_var='lick_accuracy', groupby_var=['day', 'group'], plot_transitions=[5.5, 10.5, 15.5], 
                                   transition_color=['darkgrey', 'darkgrey', 'darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=True, symbols=symbols_list,
                                   plot_datapoints=True, x_title='Day', y_title='Lick Accuracy (%)', titles=[''], height=500, width=650)
fig.add_vrect(x0=15.5, x1=16.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.update_yaxes(range=[0, 100])
fig.update_traces(name=f'mCherry ({ct_df['mouse'][ct_df['group'] == 'mCherry'].unique().shape[0]})',
                  selector={'name': 'mCherry'})
fig.update_traces(name=f'stGtACR2 ({ct_df['mouse'][ct_df['group'] == 'stGtACR2'].unique().shape[0]})',
                  selector={'name': 'stGtACR2'})
fig.show()
fig.write_image(pjoin(fig_path, 'lick_accuracy_up_to_16.png'), width=650, height=500)

## T-test for day 16 behavior
data = ct_df[ct_df['day'] == 16].reset_index(drop=True)
model = ols('lick_accuracy ~ C(group)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=3)
print(anova_table)

In [ ]:
## Plot rewards across days
fig = pf.plot_behavior_across_days(ct_df[ct_df['day'] < 21], x_var='day', y_var='rewards', groupby_var=['day', 'group'], plot_transitions=[5.5, 10.5, 15.5], 
                                   transition_color=['darkgrey', 'darkgrey', 'darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=False, symbols=symbols_list,
                                   plot_datapoints=True, x_title='Day', y_title='Rewards', titles=[''], height=500, width=650)
fig.add_vrect(x0=15.5, x1=16.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.add_vrect(x0=19.5, x1=20.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.update_traces(name=f'mCherry ({ct_df['mouse'][ct_df['group'] == 'mCherry'].unique().shape[0]})',
                  selector={'name': 'mCherry'})
fig.update_traces(name=f'stGtACR2 ({ct_df['mouse'][ct_df['group'] == 'stGtACR2'].unique().shape[0]})',
                  selector={'name': 'stGtACR2'})
fig.show()
fig.write_image(pjoin(fig_path, 'rewards.png'), width=650, height=500)

## T-test for day 16 behavior
data = ct_df[ct_df['day'] == 16].reset_index(drop=True)
model = ols('rewards ~ C(group)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=3)
print(anova_table)

In [ ]:
## Plot rewards across days including reversal
fig = pf.plot_behavior_across_days(ct_df, x_var='day', y_var='rewards', groupby_var=['day', 'group'], plot_transitions=[5.5, 10.5, 15.5, 20.5], 
                                   transition_color=['darkgrey', 'darkgrey', 'darkgrey', 'darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=False, symbols=symbols_list,
                                   plot_datapoints=True, x_title='Day', y_title='Rewards', titles=[''], height=500, width=650)
fig.add_vrect(x0=15.5, x1=16.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.add_vrect(x0=19.5, x1=20.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.add_vrect(x0=20.5, x1=21.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.update_traces(name=f'mCherry ({ct_df['mouse'][ct_df['group'] == 'mCherry'].unique().shape[0]})',
                  selector={'name': 'mCherry'})
fig.update_traces(name=f'stGtACR2 ({ct_df['mouse'][ct_df['group'] == 'stGtACR2'].unique().shape[0]})',
                  selector={'name': 'stGtACR2'})
fig.show()
fig.write_image(pjoin(fig_path, 'rewards_with_reversal.png'), width=650, height=500)

## T-test for day 16 behavior
data = ct_df[ct_df['day'] == 16].reset_index(drop=True)
model = ols('rewards ~ C(group)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=3)
print(anova_table)

In [ ]:
## Plot rewards across days up to day 16
fig = pf.plot_behavior_across_days(ct_df[ct_df['day'] < 17], x_var='day', y_var='rewards', groupby_var=['day', 'group'], plot_transitions=[5.5, 10.5, 15.5], 
                                   transition_color=['darkgrey', 'darkgrey', 'darkgrey'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=False, symbols=symbols_list,
                                   plot_datapoints=True, x_title='Day', y_title='Rewards', titles=[''], height=500, width=650)
fig.add_vrect(x0=15.5, x1=16.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.update_traces(name=f'mCherry ({ct_df['mouse'][ct_df['group'] == 'mCherry'].unique().shape[0]})',
                  selector={'name': 'mCherry'})
fig.update_traces(name=f'stGtACR2 ({ct_df['mouse'][ct_df['group'] == 'stGtACR2'].unique().shape[0]})',
                  selector={'name': 'stGtACR2'})
fig.show()
fig.write_image(pjoin(fig_path, 'rewards_up_to_day_16.png'), width=650, height=500)

## T-test for day 16 behavior
data = ct_df[ct_df['day'] == 16].reset_index(drop=True)
model = ols('rewards ~ C(group)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=3)
print(anova_table)

In [ ]:
## Look at number of licks between both groups on day 16
avg = ct_df.groupby(['group', 'day'], as_index=False).agg({'num_licks': ['mean', 'sem']})

rs = RandomState(MT19937(SeedSequence(24601)))
for it in np.arange(3):
    shifts = rs.uniform(-0.08, 0.08, ct_df['mouse'].unique().shape[0])

fig = pf.custom_graph_template(x_title='Day', y_title='Number of Licks', width=550)
for group in group_list:
    gdata = avg[(avg['group'] == group) & (avg['day'] == 16)]
    fig.add_trace(go.Scattergl(x=gdata['day'], y=gdata['num_licks']['mean'], mode='markers', marker_size=9,
                               marker=dict(line=dict(width=1.5, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['num_licks']['sem'], thickness=2.5), marker_color=group_colors_dict[group]))
for idx, mouse in enumerate(ct_df['mouse'].unique()):
    mdata = ct_df[(ct_df['mouse'] == mouse) & (ct_df['day'] == 16)]
    fig.add_trace(go.Scattergl(x=mdata['day'].values + shifts[idx], y=mdata['num_licks'], mode='markers', marker_size=6,
                               marker=dict(line=dict(width=2, color=group_colors_dict[mdata['group'].unique()[0]])),
                               name=mouse, showlegend=False, marker_color='rgba(0, 0, 0, 0)'))
fig.update_xaxes(range=[15.5, 16.5])
fig.update_xaxes(tickmode='array', tickvals=[16], ticktext=['16'])
fig.update_traces(name=f'mCherry ({ct_df['mouse'][ct_df['group'] == 'mCherry'].unique().shape[0]})',
                  selector={'name': 'mCherry'})
fig.update_traces(name=f'stGtACR2 ({ct_df['mouse'][ct_df['group'] == 'stGtACR2'].unique().shape[0]})',
                  selector={'name': 'stGtACR2'})
fig.show()
fig.write_image(pjoin(fig_path, f'day_16_number_of_licks.png'), width=550, height=500)

## T-test for day 16 behavior
data = ct_df[ct_df['day'] == 16].reset_index(drop=True)
model = ols('num_licks ~ C(group)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=3)
print(anova_table)

In [ ]:
## Look at average amount of opto stim between both groups on day x
day = 16
avg = ct_df.groupby(['group', 'day'], as_index=False).agg({'opto_time': ['mean', 'sem']})

rs = RandomState(MT19937(SeedSequence(24601)))
for it in np.arange(3):
    shifts = rs.uniform(-0.08, 0.08, ct_df['mouse'].unique().shape[0])

fig = pf.custom_graph_template(x_title='Day', y_title='Light Stimulation (s)', width=550)
for group in group_list:
    gdata = avg[(avg['group'] == group) & (avg['day'] == day)]
    fig.add_trace(go.Scattergl(x=gdata['day'], y=gdata['opto_time']['mean'], mode='markers', marker_size=9,
                               marker=dict(line=dict(width=1.5, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['opto_time']['sem'], thickness=2.5), marker_color=group_colors_dict[group]))
for idx, mouse in enumerate(ct_df['mouse'].unique()):
    mdata = ct_df[(ct_df['mouse'] == mouse) & (ct_df['day'] == day)]
    fig.add_trace(go.Scattergl(x=mdata['day'].values + shifts[idx], y=mdata['opto_time'], mode='markers', marker_size=6,
                               marker=dict(line=dict(width=2, color=group_colors_dict[mdata['group'].unique()[0]])),
                               name=mouse, showlegend=False, marker_color='rgba(0, 0, 0, 0)'))
fig.update_xaxes(range=[day - 0.5, day + 0.5])
fig.update_xaxes(tickmode='array', tickvals=[day], ticktext=[f'{day}'])
fig.update_traces(name=f'mCherry ({ct_df['mouse'][ct_df['group'] == 'mCherry'].unique().shape[0]})',
                  selector={'name': 'mCherry'})
fig.update_traces(name=f'stGtACR2 ({ct_df['mouse'][ct_df['group'] == 'stGtACR2'].unique().shape[0]})',
                  selector={'name': 'stGtACR2'})
fig.show()
fig.write_image(pjoin(fig_path, f'day_{day}_opto_time.png'), width=550, height=500)

## T-test for day x behavior
data = ct_df[ct_df['day'] == day].reset_index(drop=True)
model = ols('opto_time ~ C(group)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=3)
print(anova_table)

In [ ]:
## Plot percentage of time in the correct direction each day without reversal
fig = pf.plot_behavior_across_days(ct_df[ct_df['day'] < 21], x_var='day', y_var='correct_dir', groupby_var=['day', 'group'],
                                   marker_color=two_group_colors, avg_color=avg_color, expert_line=False, chance=False, plot_transitions=[5.5, 10.5, 15.5],
                                   transition_color=['darkgrey', 'darkgrey', 'darkgrey'], symbols=symbols_list,
                                   plot_datapoints=True, x_title='Day', y_title='Correct Direction (%)', titles=[''], height=500, width=650)
fig.add_vrect(x0=15.5, x1=16.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.add_vrect(x0=19.5, x1=20.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.update_yaxes(range=[0, 100])
fig.show()
fig.write_image(pjoin(fig_path, 'correct_dir.png'), width=650, height=500)

## Perform a linear regression to test for differences between groups. 
data = ct_df[ct_df['day'] == 16].reset_index(drop=True)
model = ols('correct_dir ~ C(group)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=3)
print(anova_table)

### Look at probe accuracy.

In [ ]:
lick_dict_probe = {'mouse': [], 'experiment': [], 'sex': [], 'group': [], 'session': [], 
                   'day': [], 'num_licks': [], 'probe_acc': [], 'session_acc': [], 'rewards': []}
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass 
    else:
        mpath = pjoin(circle_path, mouse)
        sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
        group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
        for idx, session in enumerate(natsorted(os.listdir(mpath))):
            behav = pd.read_feather(pjoin(mpath, session))
            if any(behav['probe']):
                behav_probe = behav[behav['probe']]
                behav_no_probe = behav[~behav['probe']]
                reward_one, reward_two = np.unique(behav['reward_one'])[0], np.unique(behav['reward_two'])[0]
                percent_correct = ctb.lick_accuracy(behav_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
                session_pc = ctb.lick_accuracy(behav_no_probe, port_list=[reward_one, reward_two], lick_threshold=5, by_trials=False)
                lick_dict_probe['mouse'].append(mouse)
                lick_dict_probe['experiment'].append(behav['cohort'].unique()[0])
                lick_dict_probe['sex'].append(sex)
                lick_dict_probe['group'].append(group)
                lick_dict_probe['day'].append(idx+1)
                lick_dict_probe['session'].append(np.unique(behav['session_two'])[0])
                if session == 'rto11_10.feat':
                    lick_dict_probe['num_licks'].append(np.nan)
                    lick_dict_probe['probe_acc'].append(np.nan)
                    lick_dict_probe['session_acc'].append(np.nan)
                    lick_dict_probe['rewards'].append(np.nan)
                else:
                    lick_dict_probe['num_licks'].append(len(behav_probe[behav_probe['lick_port'] != -1]))
                    lick_dict_probe['probe_acc'].append(percent_correct)
                    lick_dict_probe['session_acc'].append(session_pc)
                    lick_dict_probe['rewards'].append(np.sum(behav_no_probe['water']))
            else:
                pass
probe_df = pd.DataFrame(lick_dict_probe)
probe_df = probe_df[~pd.isna(probe_df['probe_acc'])]

In [ ]:
## Plot probe performance for first and last day in A
avg_probe = probe_df.groupby(['day', 'group'], as_index=False).agg({'probe_acc': ['mean', 'sem']})
fig = pf.custom_graph_template(x_title='Day', y_title='Lick Accuracy (%)', titles=[''])

for group in group_list:
    gdata = avg_probe[avg_probe['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['day'], y=gdata['probe_acc']['mean'], mode='markers', marker_color=group_colors_dict[group],
                               marker_size=9, marker=dict(line=dict(width=1.5, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['probe_acc']['sem'], thickness=2.5)))
fig.add_hline(y=25, line_width=3, line_dash='dash', line_color=chance_color, opacity=1, layer='below')
fig.update_yaxes(range=[0, 100])
fig.show()

In [ ]:
## Plot probe accuracy on day 17 (30-second probe)
sub_df = probe_df[probe_df['day'] == 17]
avg = sub_df.groupby(['group', 'day'], as_index=False).agg({'probe_acc': ['mean', 'sem']})
fig = pf.custom_graph_template(x_title='Day', y_title='Lick Accuracy (%)', width=550)

rs = RandomState(MT19937(SeedSequence(24601)))
for it in np.arange(4):
    shifts = rs.uniform(-0.08, 0.08, ct_df['mouse'].unique().shape[0])

for group in group_list:
    gdata = avg[avg['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['day'], y=gdata['probe_acc']['mean'], mode='markers', marker_color=group_colors_dict[group],
                               marker_size=9, marker=dict(line=dict(width=1.5, color='black')), name=group,
                               error_y=dict(type='data', array=gdata['probe_acc']['sem'], thickness=2.5)))

for idx, mouse in enumerate(sub_df['mouse'].unique()):
    mdata = sub_df[sub_df['mouse'] == mouse]
    fig.add_trace(go.Scattergl(x=mdata['day'] + shifts[idx], y=mdata['probe_acc'], mode='markers', marker_size=6,
                               marker=dict(line=dict(width=2, color=group_colors_dict[mdata['group'].unique()[0]])), name=mouse, showlegend=False,
                               marker_color='rgba(0, 0, 0, 0)'))
fig.update_xaxes(range=[16.5, 17.5])
fig.update_xaxes(tickmode='array', tickvals=[17], ticktext=['17'])
fig.add_hline(y=25, line_width=3, line_dash='dash', line_color=chance_color, opacity=1)
fig.update_traces(name=f'mCherry ({sub_df['mouse'][sub_df['group'] == 'mCherry'].unique().shape[0]})',
                  selector={'name': 'mCherry'})
fig.update_traces(name=f'stGtACR2 ({sub_df['mouse'][sub_df['group'] == 'stGtACR2'].unique().shape[0]})',
                  selector={'name': 'stGtACR2'})
fig.show()
fig.write_image(pjoin(fig_path, 'day_17_30s_probe_acc.png'), width=550, height=500)

### Look at accuracy across trials.

In [ ]:
bin_size = 4
lick_thresh = 5
trial_res = {'mouse': [], 'sex': [], 'group': [], 'day': [], 'session_two': [], 'trial': [], 'lick_acc': [], 'max_trial': []}
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass 
    else:
        mpath = pjoin(circle_path, mouse)
        sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
        group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
        for idx, session in enumerate(natsorted(os.listdir(mpath))):
            behav = pd.read_feather(pjoin(mpath, f'{session}'))
            reward_one, reward_two = behav['reward_one'].unique()[0], behav['reward_two'].unique()[0]
            trial_acc = ctb.lick_accuracy(behav, port_list=[reward_one, reward_two], lick_threshold=lick_thresh, by_trials=True)
            max_trial = np.max(behav['trials'])
            
            if bin_size > 1:
                binned_acc = ctb.bin_data(trial_acc, bin_size)
            else:
                binned_acc = trial_acc

            if session == 'rto11_10.feat':
                pass 
            else:
                for trial, val in enumerate(binned_acc):
                    trial_res['mouse'].append(mouse)
                    trial_res['sex'].append(sex)
                    trial_res['group'].append(group)
                    trial_res['day'].append(idx+1)
                    trial_res['session_two'].append(behav['session_two'].unique()[0])
                    trial_res['trial'].append(trial * bin_size)
                    trial_res['lick_acc'].append(val)
                    trial_res['max_trial'].append(max_trial)
trial_df = pd.DataFrame(trial_res)

In [ ]:
## Plot lick accuracy across trials on day x
day = 16
match_trials = True 

day_data = trial_df[trial_df['day'] == day]
if match_trials:
    sub_df = day_data[day_data['trial'] <= np.min(day_data['max_trial'])]
else:
    sub_df = day_data.copy()
avg = sub_df.groupby(['trial', 'group', 'session_two'], as_index=False).agg({'lick_acc': ['mean', 'sem']})
avg = avg[~pd.isna(avg['lick_acc']['sem'])]

fig = pf.custom_graph_template(x_title='Trial', y_title='Lick Accuracy (%)', titles=[f'Day {day}'], width=650)
for group in group_list:
    gdata = avg[avg['group'] == group]
    xaxis = np.arange(1, gdata['trial'].to_numpy()[-1] * bin_size, bin_size)
    upper = gdata['lick_acc']['mean'] + gdata['lick_acc']['sem']
    lower = gdata['lick_acc']['mean'] - gdata['lick_acc']['sem']

    fig.add_trace(go.Scattergl(x=xaxis, y=gdata['lick_acc']['mean'], mode='lines', line_color=group_colors_dict[group],
                                   name=group, showlegend=True, legendgroup=group))
    fig.add_trace(go.Scatter(x=xaxis, y=upper, mode='lines', marker=dict(color=error_dict[group]),
                                    name='Upper Bound', line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=xaxis, y=lower, mode='lines', marker=dict(color=error_dict[group]),
                            name='Lower Bound', line=dict(width=0), showlegend=False, fillcolor=error_dict[group], fill='tonexty'))
fig.add_hline(y=25, line_width=3, line_dash='dash', line_color='darkgrey', opacity=1, layer='below')
fig.update_yaxes(range=[0, 101])
fig.update_traces(name=f'mCherry ({trial_df['mouse'][trial_df['group'] == 'mCherry'].unique().shape[0]})',
                  selector={'name': 'mCherry'})
fig.update_traces(name=f'stGtACR2 ({trial_df['mouse'][trial_df['group'] == 'stGtACR2'].unique().shape[0]})',
                  selector={'name': 'stGtACR2'})
fig.show()
fig.write_image(pjoin(fig_path, f'accuracy_across_trials_day{day}.png'), width=650, height=500)

## Repeated measures ANOVA
data = sub_df[sub_df['day'] == day]
data.mixed_anova(dv='lick_acc', within='trial', between='group', subject='mouse')

In [ ]:
## Plot individual trial by trial learning curves for x day
day = 16
fig = pf.custom_graph_template(x_title='', y_title='', rows=1, columns=2, shared_x=True, shared_y=True,
                               titles=['mCherry', 'stGtACR2'], width=1000)

for mouse in trial_df['mouse'].unique():
    mdata = trial_df[(trial_df['mouse'] == mouse) & (trial_df['day'] == day)]
    if mdata['group'].unique()[0] == 'mCherry':
        col = 1
        g = 'mCherry'
    else:
        col = 2
        g = 'stGtACR2'

    fig.add_trace(go.Scattergl(x=mdata['trial'] * bin_size, y=mdata['lick_acc'], mode='lines', line_color=group_colors_dict[g],
                               name=mouse, line_width=2), row=1, col=col)
fig.add_hline(y=25, line_width=3, line_dash='dash', line_color='darkgrey', opacity=1, layer='below')
fig.show()
fig.write_image(pjoin(fig_path, f'day_{day}_individual_trial_acc_curves.png'), width=1000, height=500)

### Signal detection metrics.

In [ ]:
## Get signal detection metrics (dprime, correct rejection rate, hit rate)
## Get signal detection metrics binned across trials (set by bin_size)
bin_size = 4 ## in trials

sig_df = pd.DataFrame()
for mouse in os.listdir(circle_path):
    if mouse in excluded_mice:
        pass 
    else:
        mpath = pjoin(circle_path, mouse)
        sex = maze_info['Sex'][maze_info['Mouse'] == mouse].values[0]
        group = maze_info['Group'][maze_info['Mouse'] == mouse].values[0]
        for idx, session in enumerate(natsorted(os.listdir(mpath))):
            behav = pd.read_feather(pjoin(mpath, f'{session}'))
            reward_one, reward_two = behav['reward_one'].unique()[0], behav['reward_two'].unique()[0]
            signal = pd.DataFrame(ctb.dprime_metrics(behav, mouse, day=idx + 1, reward_ports=[reward_one, reward_two], forward_reverse='all'))
            signal['experiment'] = behav['cohort'].unique()[0]
            signal['group'] = group 
            signal['sex'] = sex
            sig_df = pd.concat([sig_df, signal], ignore_index=True)
avg_sig = sig_df.groupby(['day', 'mouse', 'group'], as_index=False).agg({'CR': 'mean', 'FA': 'mean', 'dprime': 'mean', 'hits': 'mean'})

binned_dict = {'mouse': [], 'group': [], 'day': [], 'trial': [], 'max_trial': [], 'hits': [], 'CR': []}
for mouse in sig_df['mouse'].unique():
    mdata = sig_df[sig_df['mouse'] == mouse]
    for day in mdata['day'].unique():
        d_data = mdata[mdata['day'] == day]
        max_trial = np.max(d_data['trial'])
        bins = np.arange(0, d_data['trial'].to_numpy()[-1] + bin_size, bin_size)
        for idx, (b_start, b_end) in enumerate(zip(bins[:-1], bins[1:])):
            sub = d_data[(d_data['trial'] > b_start) & (d_data['trial'] <= b_end)]
            if sub.shape[0] < bin_size:
                pass 
            else:
                binned_dict['mouse'].append(mouse)
                binned_dict['group'].append(mdata['group'].unique()[0])
                binned_dict['day'].append(day)
                binned_dict['trial'].append(idx * bin_size)
                binned_dict['max_trial'].append(max_trial)
                binned_dict['hits'].append(np.mean(sub['hits'].to_numpy()))
                binned_dict['CR'].append(np.mean(sub['CR'].to_numpy()))
binned_df = pd.DataFrame(binned_dict)

In [ ]:
## Plot hit rate across days
avg = avg_sig.groupby(['group', 'day'], as_index=False).agg({'CR': ['mean', 'sem'], 'FA': ['mean', 'sem'], 'dprime': ['mean', 'sem'], 'hits': ['mean', 'sem']})
fig = pf.custom_graph_template(x_title='Day', y_title='Hit Rate', width=600)
for group in group_list:
    gdata = avg[avg['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['day'], y=gdata['hits']['mean'], mode='lines+markers', marker_size=9, 
                               marker=dict(line=dict(width=1.5, color='black')), line_color=group_colors_dict[group],
                               error_y=dict(type='data', array=gdata['hits']['sem'], thickness=2.5), name=group))
fig.add_vrect(x0=15.5, x1=16.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.add_vrect(x0=19.5, x1=20.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
for val in [5.5, 10.5, 15.5]:
    fig.add_vline(x=val, line_dash='dash', line_color=chance_color, opacity=1, line_width=3, layer='below')
fig.update_yaxes(range=[0, 1])
fig.show()
fig.write_image(pjoin(fig_path, 'hit_rate.png'), width=600, height=500)

## T-test for day 16 behavior
data = avg_sig[avg_sig['day'] == 16].reset_index(drop=True)
model = ols('hits ~ C(group)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=3)
print(anova_table)

In [ ]:
## Plot correct rejection rate across days
avg = avg_sig.groupby(['group', 'day'], as_index=False).agg({'CR': ['mean', 'sem'], 'FA': ['mean', 'sem'], 'dprime': ['mean', 'sem'], 'hits': ['mean', 'sem']})
fig = pf.custom_graph_template(x_title='Day', y_title='Correct Rejection Rate', width=600)
for group in group_list:
    gdata = avg[avg['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['day'], y=gdata['CR']['mean'], mode='lines+markers', marker_size=9, 
                               marker=dict(line=dict(width=1.5, color='black')), line_color=group_colors_dict[group],
                               error_y=dict(type='data', array=gdata['CR']['sem'], thickness=2.5), name=group))
fig.add_vrect(x0=15.5, x1=16.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.add_vrect(x0=19.5, x1=20.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
for val in [5.5, 10.5, 15.5]:
    fig.add_vline(x=val, line_dash='dash', line_color=chance_color, opacity=1, line_width=3, layer='below')
fig.update_yaxes(range=[0, 1])
fig.show()
fig.write_image(pjoin(fig_path, 'correct_rejection_rate.png'), width=600, height=500)

## T-test for day 16 behavior
data = avg_sig[avg_sig['day'] == 16].reset_index(drop=True)
model = ols('CR ~ C(group)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=3)
print(anova_table)

In [ ]:
## Plot dprime across days
avg = avg_sig.groupby(['group', 'day'], as_index=False).agg({'CR': ['mean', 'sem'], 'FA': ['mean', 'sem'], 'dprime': ['mean', 'sem'], 'hits': ['mean', 'sem']})
fig = pf.custom_graph_template(x_title='Day', y_title="d'", width=600)
for group in group_list:
    gdata = avg[avg['group'] == group]
    fig.add_trace(go.Scattergl(x=gdata['day'], y=gdata['dprime']['mean'], mode='lines+markers', marker_size=9, 
                               marker=dict(line=dict(width=1.5, color='black')), line_color=group_colors_dict[group],
                               error_y=dict(type='data', array=gdata['dprime']['sem'], thickness=2.5), name=group))
fig.add_vrect(x0=15.5, x1=16.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
fig.add_vrect(x0=19.5, x1=20.5, fillcolor='blue', layer='below', opacity=0.4, line_width=0)
for val in [5.5, 10.5, 15.5]:
    fig.add_vline(x=val, line_dash='dash', line_color=chance_color, opacity=1, line_width=3, layer='below')
fig.show()
fig.write_image(pjoin(fig_path, 'dprime.png'), width=600, height=500)

## T-test for day 16 behavior
data = avg_sig[avg_sig['day'] == 16].reset_index(drop=True)
model = ols('dprime ~ C(group)', data=data).fit()
anova_table = sm.stats.anova_lm(model, typ=3)
print(anova_table)

In [ ]:
## Plot hit rate across trials on day x
day = 16
match_trials = True 

day_data = binned_df[binned_df['day'] == day]
if match_trials:
    sub_df = day_data[day_data['trial'] <= np.min(day_data['max_trial'])]
else:
    sub_df = day_data.copy()
trial_sig = sub_df.groupby(['trial', 'group'], as_index=False).agg({'hits': ['mean', 'sem']})
trial_sig = trial_sig[~pd.isna(trial_sig['hits']['sem'])]

fig = pf.custom_graph_template(x_title='Trial', y_title='Hit Rate', width=600)
for group in group_list:
    gdata = trial_sig[trial_sig['group'] == group]
    xaxis = gdata['trial']
    upper = gdata['hits']['mean'] + gdata['hits']['sem']
    lower = gdata['hits']['mean'] - gdata['hits']['sem']
    fig.add_trace(go.Scattergl(x=xaxis, y=gdata['hits']['mean'], mode='lines', line_color=group_colors_dict[group],
                               line_width=2, showlegend=True, name=group, legendgroup=group))
    fig.add_trace(go.Scatter(x=xaxis, y=upper, mode='lines', marker=dict(color=error_dict[group]),
                               name='Upper Bound', line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(x=xaxis, y=lower, mode='lines', marker=dict(color=error_dict[group]),
                               name='Lower Bound', line=dict(width=0), showlegend=False, fillcolor=error_dict[group], fill='tonexty'))
fig.update_traces(name=f'mCherry ({sub_df['mouse'][sub_df['group'] == 'mCherry'].unique().shape[0]})',
                  selector={'name': 'mCherry'})
fig.update_traces(name=f'stGtACR2 ({sub_df['mouse'][sub_df['group'] == 'stGtACR2'].unique().shape[0]})',
                  selector={'name': 'stGtACR2'})
fig.show()
fig.write_image(pjoin(fig_path, f'day_{day}_hr_across_trials.png'), width=600, height=500)

## Repeated measures ANOVA
data = sub_df[sub_df['day'] == day]
data.mixed_anova(dv='hits', within='trial', between='group', subject='mouse')